# Basic Usage


# Benchmarking the Nuts Experimentalist vs. Random Sampling

This notebook demonstrates how to benchmark the performance of the **Nuts Experimentalist** against a standard **Random Sampler**.

The goal is to see which sampling strategy helps a "theorist" learn the underlying "ground truth" of a scientific model more efficiently. A more efficient experimentalist will guide the theorist to a lower error (MSE) with fewer experimental samples.

We will simulate this process for two different ground truth models:
1.  **Exponential Learning Model**
2.  **Luce Choice Ratio Model**

For our theorist, we will use the **Bayesian Machine Scientist (BMS)** in all simulations.

In [ ]:
%%capture
# Install standard autora packages
!pip install "autora[theorist-bms]"

# Install your Nuts experimentalist from TestPyPI
!pip install --upgrade --no-cache-dir --index-url https://test.pypi.org/simple --extra-index-url https://pypi.org/simple autora-experimentalist-challenge-nuts

In [ ]:
# General Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# AutoRA State and Variables
from autora.state import StandardState, on_state, Delta
from autora.variable import VariableCollection

# Ground Truth Models (Experiment Runners)
from autora.experiment_runner.synthetic.psychology.exp_learning import exp_learning
from autora.experiment_runner.synthetic.psychology.luce_choice_ratio import luce_choice_ratio

# Theorist
from autora.theorist.bms import BMSRegressor

# Experimentalists
from autora.experimentalist.grid import grid_pool
from autora.experimentalist.random import random_sample
from autora_experimentalist_challenge_nuts import sample as nuts_sample

## Defining the Workflow Components

Here we define the wrappers for our theorist and experimentalists. These functions allow the components to interact with the AutoRA `StandardState` object, which holds all the data and models for a given simulation.

In [ ]:
# 1. Theorist Wrapper
@on_state("experiment_data")
def bms_theorist(experiment_data: pd.DataFrame, bms_epochs: int = 100):
    """ Fits the BMS theorist to the experimental data. """
    ivs = experiment_data.columns[:-1]
    dvs = experiment_data.columns[-1]
    X = experiment_data[ivs]
    y = experiment_data[dvs]
    
    model = BMSRegressor(epochs=bms_epochs)
    model.fit(X, y)
    
    return Delta(models=[model])

# 2. Experimentalist Wrappers
@on_state("conditions", "models", "experiment_data")
def nuts_sampler_on_state(conditions, models, experiment_data, num_samples=1):
    """ Wrapper for your Nuts Experimentalist. """
    reference_conditions = experiment_data[conditions.columns]
    return Delta(conditions=nuts_sample(
        conditions=conditions,
        models=models,
        reference_conditions=reference_conditions,
        num_samples=num_samples
    ))

@on_state("conditions")
def random_sampler_on_state(conditions, num_samples=1):
    """ Wrapper for the Random Sampler. """
    return Delta(conditions=random_sample(conditions, num_samples=num_samples))


# 3. Experiment Runner Wrapper
@on_state("conditions")
def synthetic_runner_on_state(conditions, experiment_runner):
    """ Wrapper for the synthetic experiment runner. """
    new_data = experiment_runner.run(conditions, added_noise=0.01)
    return Delta(experiment_data=new_data)

## Simulation and Evaluation Functions

Next, we define two key functions:
- `evaluate_model`: Calculates the Mean Squared Error (MSE) of our theorist's model against the true ground truth.
- `run_benchmark`: Runs a full closed-loop simulation for a given experimentalist and ground truth.

In [ ]:
def evaluate_model(model, ground_truth_runner):
    """ Calculates the MSE of a model against a validation set from the ground truth. """
    # Create a dense grid of points across the design space for validation
    validation_conditions = grid_pool(ground_truth_runner.variables)
    
    # Get the "true" observations from the ground truth model
    true_observations = ground_truth_runner.run(validation_conditions)
    
    # Get the theorist's predictions for these conditions
    model_predictions = model.predict(validation_conditions)
    
    # Calculate Mean Squared Error
    mse = np.mean((true_observations.performance - model_predictions.flatten())**2)
    return mse

def run_benchmark(experimentalist_func, ground_truth_runner, num_cycles=20, bms_epochs=100):
    """
    Runs a full simulation loop for a given experimentalist and ground truth.
    
    Returns: A list of MSE values, one for each cycle.
    """
    
    # Initialize the state with 3 random samples
    initial_conditions = random_sample(grid_pool(ground_truth_runner.variables), num_samples=3)
    state = StandardState(
        variables=ground_truth_runner.variables,
        conditions=initial_conditions
    )
    state = synthetic_runner_on_state(state, experiment_runner=ground_truth_runner)
    
    mse_log = []

    for i in range(num_cycles):
        print(f"Running cycle {i+1}/{num_cycles}...")
        # 1. Theorist Phase
        state = bms_theorist(state, bms_epochs=bms_epochs)
        
        # 2. Evaluation Phase
        mse = evaluate_model(state.models[-1], ground_truth_runner)
        mse_log.append(mse)
        
        # 3. Experimentalist Phase
        all_conditions = grid_pool(state.variables)
        state = experimentalist_func(state, conditions=all_conditions)
        
        # 4. Experiment Runner Phase
        state = synthetic_runner_on_state(state, experiment_runner=ground_truth_runner)
        
    return mse_log

## Benchmark 1: Exponential Learning Model

Now, we run the benchmark for the `exp_learning` ground truth model. We will run one simulation with the Random Sampler and one with the Nuts Experimentalist.

In [ ]:
# Define the ground truth for this benchmark
exp_learning_runner = exp_learning()

# Run the simulation for the Random Sampler
print("--- Starting Benchmark for Random Sampler ---")
mse_log_random_exp = run_benchmark(random_sampler_on_state, exp_learning_runner)

# Run the simulation for your Nuts Experimentalist
print("\n--- Starting Benchmark for Nuts Experimentalist ---")
mse_log_nuts_exp = run_benchmark(nuts_sampler_on_state, exp_learning_runner)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mse_log_random_exp, label="Random Sampler", marker='o', linestyle='--')
plt.plot(mse_log_nuts_exp, label="Nuts Experimentalist", marker='x', linestyle='-')
plt.title("Benchmark 1: Exponential Learning Model Performance", fontsize=16)
plt.xlabel("Number of Experiments Sampled", fontsize=12)
plt.ylabel("Mean Squared Error (MSE)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

### Analysis 1

The plot above shows the learning curve for the BMS theorist when guided by each experimentalist. A steeper downward slope indicates a more efficient sampling strategy, as the theorist is learning the ground truth faster (lower MSE) with fewer samples.

## Benchmark 2: Luce Choice Ratio Model

Next, we repeat the process for a different, more complex ground truth: the `luce_choice_ratio` model.

In [ ]:
# Define the ground truth for this benchmark
luce_choice_runner = luce_choice_ratio()

# Run the simulation for the Random Sampler
print("--- Starting Benchmark for Random Sampler ---")
mse_log_random_luce = run_benchmark(random_sampler_on_state, luce_choice_runner)

# Run the simulation for your Nuts Experimentalist
print("\n--- Starting Benchmark for Nuts Experimentalist ---")
mse_log_nuts_luce = run_benchmark(nuts_sampler_on_state, luce_choice_runner)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mse_log_random_luce, label="Random Sampler", marker='o', linestyle='--')
plt.plot(mse_log_nuts_luce, label="Nuts Experimentalist", marker='x', linestyle='-')
plt.title("Benchmark 2: Luce Choice Ratio Model Performance", fontsize=16)
plt.xlabel("Number of Experiments Sampled", fontsize=12)
plt.ylabel("Mean Squared Error (MSE)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

### Analysis 2

This second benchmark tests the experimentalists against a different kind of underlying model. Comparing the results from both benchmarks helps evaluate how robust each sampling strategy is across different scientific problems.